In [0]:
files = dbutils.fs.ls("s3://swift-pipeline-aws/bronze/")
for f in files[:5]:
  print(f.name)

In [0]:
from pyspark.sql.functions import col, explode, when, lit, array_contains, substring
import time

start = time.time()

BRONZE = "s3://swift-pipeline-aws/bronze/ingest_date=*"
SCHEMA = "dataengineering.swift"


def defect_rule(party):
    town = col(f"{party}_town")
    country = col(f"{party}_country")
    lines = col(f"{party}_adr_lines")
    return (
        when(town.isNull() & lines.isNotNull(), lit("UNSTRUCTURED_ONLY"))
        .when(country.isNull() & town.isNotNull(), lit("MISSING_COUNTRY"))
        .when(town.isNull() & country.isNotNull(), lit("MISSING_TOWN"))
        .when(country == "NOTPROVIDED", lit("PLACEHOLDER_COUNTRY"))
        .when(country == "UK", lit("INVALID_COUNTRY"))
        .when(lines.isNotNull() & array_contains(lines, town), lit("DUPLICATED_TOWN"))
        .otherwise(lit("CLEAN"))
    )


raw = spark.read.format("xml") \
    .option("rowTag", "FIToFICstmrCdtTrf") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt") \
    .load(BRONZE)

raw.write.mode("overwrite").saveAsTable(f"{SCHEMA}.bronze_parsed")
raw = spark.table(f"{SCHEMA}.bronze_parsed")

dead_letter = raw.filter(col("_corrupt").isNotNull())
good = raw.filter(col("_corrupt").isNull())

silver = good.select(
    col("GrpHdr.MsgId").alias("msg_id"),
    col("GrpHdr.CreDtTm").alias("creation_dt"),
    explode(col("CdtTrfTxInf")).alias("tx")
).select(
    "msg_id",
    "creation_dt",
    col("tx.PmtId.UETR").alias("uetr"),
    col("tx.PmtId.EndToEndId").alias("end_to_end_id"),
    col("tx.IntrBkSttlmAmt._VALUE").alias("amount"),
    col("tx.IntrBkSttlmAmt._Ccy").alias("currency"),
    col("tx.InstdAmt._VALUE").alias("amount_cad"),
    col("tx.PmtTpInf.LclInstrm.Cd").alias("channel"),
    col("tx.Dbtr.Nm").alias("debtor_name"),
    col("tx.DbtrAgt.FinInstnId.BICFI").alias("debtor_bic"),
    col("tx.Dbtr.PstlAdr.StrtNm").alias("debtor_street"),
    col("tx.Dbtr.PstlAdr.PstCd").alias("debtor_postcode"),
    col("tx.Dbtr.PstlAdr.TwnNm").alias("debtor_town"),
    col("tx.Dbtr.PstlAdr.Ctry").alias("debtor_country"),
    col("tx.Dbtr.PstlAdr.AdrLine").alias("debtor_adr_lines"),
    col("tx.Cdtr.Nm").alias("creditor_name"),
    col("tx.CdtrAgt.FinInstnId.BICFI").alias("creditor_bic"),
    col("tx.CdtrAcct.Id.IBAN").alias("creditor_iban"),
    col("tx.Cdtr.PstlAdr.StrtNm").alias("creditor_street"),
    col("tx.Cdtr.PstlAdr.PstCd").alias("creditor_postcode"),
    col("tx.Cdtr.PstlAdr.TwnNm").alias("creditor_town"),
    col("tx.Cdtr.PstlAdr.Ctry").alias("creditor_country"),
    col("tx.Cdtr.PstlAdr.AdrLine").alias("creditor_adr_lines"),
)

tagged = silver \
    .withColumn("debtor_defect", defect_rule("debtor")) \
    .withColumn("creditor_defect", defect_rule("creditor")) \
    .withColumn("corridor", substring(col("creditor_bic"), 5, 2))

clean = tagged.filter((col("debtor_defect") == "CLEAN") & (col("creditor_defect") == "CLEAN"))
quarantine = tagged.filter((col("debtor_defect") != "CLEAN") | (col("creditor_defect") != "CLEAN"))

clean.write.mode("overwrite").saveAsTable(f"{SCHEMA}.silver_payments")
quarantine.write.mode("overwrite").saveAsTable(f"{SCHEMA}.silver_quarantine")
dead_letter.write.mode("overwrite").saveAsTable(f"{SCHEMA}.silver_dead_letter")

print("clean:", clean.count(), "quarantine:", quarantine.count(), "dead:", dead_letter.count())
print("seconds:", round(time.time() - start, 1))


In [0]:
raw.write.mode("overwrite").saveAsTable(f"{SCHEMA}.bronze_raw")
raw = spark.table(f"{SCHEMA}.bronze_raw")

In [0]:
%sql
DROP TABLE dataengineering.swift.bronze_parsed